# 06a - XAI Setup
Nacte vsechna data a modely potrebne pro vysvetleni. Ostatni XAI notebooky tento soubor nezacitkuji znovu.

**Zavislosti:**
- `data/processed/split_data.pkl` — preprocessovana data
- `models/best_model.pkl` — vitezny tuned model (pipeline: SMOTE + model)
- `models/tuned/*.pkl` — vsechny tri tuned modely
- `models/baseline/*.pkl` — vsechny tri baseline modely

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
import os

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

print(f"Train: {X_train_prep.shape}, Test: {X_test_prep.shape}")
print(f"Dropout rate (test): {y_test.mean():.1%}")
print(f"Priznaky ({len(X_train_prep.columns)}): {list(X_train_prep.columns)}")

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


Train: (8000, 19), Test: (2000, 19)
Dropout rate (test): 23.5%
Priznaky (19): ['Family_Income', 'Study_Hours_per_Day', 'Stress_Index', 'Age_Gap', 'CGPA', 'GPA_trend', 'Department_Arts', 'Department_Business', 'Department_CS', 'Department_Engineering', 'Department_Science', 'Gender', 'Internet_Access', 'Attendance_Rate', 'Assignment_Delay_Days', 'Travel_Time_Minutes', 'Part_Time_Job', 'Scholarship', 'Semester']


## Nacteni vsech modelu
Kazdý model je ulozeny jako pipeline (SMOTE + estimator). Pro XAI extrahujem jen samotny estimator — SMOTE je relevantni pouze pri treniku, ne pri vysvetleni.

In [2]:
def load_estimator(path):
    pipeline = joblib.load(path)
    # Tuned modely jsou ImbPipeline s krokem 'model'
    if hasattr(pipeline, 'named_steps') and 'model' in pipeline.named_steps:
        return pipeline.named_steps['model']
    # Baseline modely jsou ulozene primo jako estimatory
    return pipeline

# Tuned modely (primarni zdroj pro XAI)
tuned_models = {
    'Logistic Regression': load_estimator('../models/tuned/logistic_regression_tuned.pkl'),
    'Random Forest':       load_estimator('../models/tuned/random_forest_tuned.pkl'),
    'Gradient Boosting':   load_estimator('../models/tuned/gradient_boosting_tuned.pkl'),
    'Decision Tree':       load_estimator('../models/tuned/decision_tree_tuned.pkl'),
}

# Vitezny model (pro lokalni vysvetleni a LIME)
best_pipeline  = joblib.load('../models/best_model.pkl')
best_estimator = load_estimator('../models/best_model.pkl')

# Metadata o vitezi
import json as _json
with open('../models/best_model_metadata.json') as _f:
    _metadata = _json.load(_f)
winning_name = _metadata['winning_model']

print(f"Vitezny model: {winning_name}")
for name, model in tuned_models.items():
    print(f"  Nacten: {name} ({type(model).__name__})")


Vitezny model: Logistic Regression
  Nacten: Logistic Regression (LogisticRegression)
  Nacten: Random Forest (RandomForestClassifier)
  Nacten: Gradient Boosting (GradientBoostingClassifier)
  Nacten: Decision Tree (DecisionTreeClassifier)


## Identifikace rizikových studentů
Vybereme studenty s predikovanou pravdepodobnosti dropoutu 80–90 % — dostatecne rizikovi pro zajimave vysvetleni, ale ne trivialni pripady.

In [3]:
probabilities = best_estimator.predict_proba(X_test_prep)[:, 1]

risky_indices = [i for i, p in enumerate(probabilities) if 0.80 <= p <= 0.90]
print(f"Studentu s rizikem 80-90 %: {len(risky_indices)}")

# Primarni student pro lokalni vysvetleni
student_index  = risky_indices[0]
risky_student  = X_test_prep.iloc[student_index]
student_prob   = probabilities[student_index]

print(f"\nZvoleny student: index {student_index}, riziko {student_prob:.1%}")
print("Nenulove hodnoty:")
print(risky_student[risky_student != 0].head(10).to_string())

# --- Atribut zájmu (INTEREST_FEATURE) ---
# Vybíráme ovlivnitelný faktor — takový, který student může aktivně měnit
INTEREST_FEATURE = 'Study_Hours_per_Day'
print(f"\nAtribut zájmu (INTEREST_FEATURE): {INTEREST_FEATURE}")
print(f"Hodnota u vybraného studenta: {risky_student[INTEREST_FEATURE]:.4f}")


Studentu s rizikem 80-90 %: 150

Zvoleny student: index 17, riziko 85.9%
Nenulove hodnoty:
Family_Income         -0.083656
Study_Hours_per_Day   -2.100029
Stress_Index           0.277687
Age_Gap                1.162969
CGPA                  -1.611627
GPA_trend              0.502424
Department_Science          1.0
Gender                        1
Internet_Access               1
Attendance_Rate            87.6

Atribut zájmu (INTEREST_FEATURE): Study_Hours_per_Day
Hodnota u vybraného studenta: -2.1000


## Ulozeni sdileneho kontextu
Ostatni notebooky tento soubor nespousteji — misto toho nactou pkl s vysledky z tohoto notebooku.

In [4]:
os.makedirs('../results/xai', exist_ok=True)

joblib.dump({
    'X_train_prep':    X_train_prep,
    'X_test_prep':     X_test_prep,
    'y_train':         y_train,
    'y_test':          y_test,
    'tuned_models':    tuned_models,
    'best_estimator':  best_estimator,
    'best_pipeline':   best_pipeline,
    'winning_name':    winning_name,
    'probabilities':   probabilities,
    'student_index':   student_index,
    'risky_student':   risky_student,
    'student_prob':    student_prob,
    'INTEREST_FEATURE': INTEREST_FEATURE,
}, '../results/xai/xai_context.pkl')

print("Kontext ulozen do results/xai/xai_context.pkl")
print(f"  INTEREST_FEATURE = '{INTEREST_FEATURE}'")
print(f"  student_index    = {student_index}")


Kontext ulozen do results/xai/xai_context.pkl
  INTEREST_FEATURE = 'Study_Hours_per_Day'
  student_index    = 17
